## **Longitudinal Data Analysis of Housing Supply and Poverty Exposure in the U.S.**

By Adrian Lopez. MATH 120, Fall 2025

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [422]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/MATH120_Final_Project/'):
        !git clone https://github.com/Adrian1840/MATH120_Final_Project

    # Change to project directory
    os.chdir('/content/MATH120_Final_Project')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Current working directory: /content/MATH120_Final_Project


## Import Libraries

In [423]:
import numpy as np
import pandas as pd
import statsmodels.api as sm #for linear regression
# import plotting packages
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import zipfile

## Data Loading & Cleaning

In [424]:
path = "/content/MATH120_Final_Project/data/"

In [425]:
#overall county-lvl mobility metrics (clean, removed NA columns that made CSV file big)
county = pd.read_csv(path + "county_mobility.csv", encoding='latin-1',
    low_memory=False) #used this encoding and false low_memory to fix errors with mixed data types
county.head() #see head of the data

,year,state,county,state_name,county_name,population,pctl_income_20,pctl_income_20_lb,pctl_income_20_ub,pctl_income_20_quality,...,share_hs_degree_ub,share_hs_degree_quality,share_digital_access,share_digital_access_quality,share_employed,share_employed_lb,share_employed_ub,share_employed_quality,ratio_living_wage,ratio_living_wage_quality
0,2014,1,1,Alabama,Autauga County,55395,20783.63925,17647.4375,25210.6250,Weak,...,0.866112,Weak,NaN,NaN,0.733227,0.684302,0.782152,Weak,0.628396,Strong
1,2014,1,3,Alabama,Baldwin County,200111,19160.07500,15832.2725,23193.7750,Strong,...,0.939239,Strong,NaN,NaN,0.753256,0.692454,0.814058,Strong,0.599644,Strong
2,2014,1,5,Alabama,Barbour County,26887,14117.95000,11596.8875,16538.1700,Weak,...,0.965525,Weak,NaN,NaN,0.636348,0.589012,0.683684,Weak,0.648669,Strong
3,2014,1,7,Alabama,Bibb County,22506,10951.49550,8974.9825,12907.8400,Weak,...,0.960699,Weak,NaN,NaN,0.599087,0.529003,0.669171,Weak,0.677315,Strong
4,2014,1,9,Alabama,Blount County,57719,17143.22500,14924.6900,20672.7125,Marginal,...,0.858308,Marginal,NaN,NaN,0.703360,0.653465,0.753255,Marginal,0.596074,Strong


## Research Question 1
* How have U.S. states with the lowest 5 average affordable housing supply changed from 2014–2023?



In [427]:
#Avg Share of affordable/available housing units per 100 households w/ low incomes (80 Area Median Income (AMI)) by state and year
housing_st_yr = county.groupby(["state_name","year"], as_index=False).agg(avg_afford= ("share_affordable_80_ami","mean")) #adds pop col with mean of pop
housing_st_yr["year"] = housing_st_yr["year"].astype(int)

#Dropping NA values
housing_st_yr = housing_st_yr.dropna()
housing_st_yr.head()

,state_name,year,avg_afford
0,Alabama,2014,1.738907
1,Alabama,2015,1.760706
2,Alabama,2016,1.846129
3,Alabama,2017,2.022749
4,Alabama,2018,1.920754


In [428]:
#Gets a vector (.index) of the 5 states with the lowest average affordability supply
lowest_5_hous = housing_st_yr.groupby("state_name").mean().sort_values(by="avg_afford",ascending=True).head(5).index

#Gets subset of the avg_st_yr dataframe but only selects the states in the lowest_5
avg_low = housing_st_yr[housing_st_yr["state_name"].isin(lowest_5_hous)]

subtitle="Selected 5 U.S. States with the Lowest Average Housing Affordability Supply" #Subtitle will stay same in all plots

#Plotting Avg affordable housing units per 100 households w/ extremely low incomes over years
fig = px.line(
    avg_low,
    x="year",
    y="avg_afford",
    color="state_name",
    markers=True,
      title=(
        "Average Affordable Housing Units per 100 Households Over Time"
        f"<br><span style='font-size:15px;'>{subtitle}</span>" # subtitle uses HTML
    )
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Averge Units per 100 Households",
    legend_title_text="State"
)


fig.show()

* This graph looks at the 5 states in the United States that have the *lowest* average share of affordable housing units per 100 households with low incomes.
    * These households of low incomes are specifically below 80% of the Area Median Income (AMI).

## Research Question 2
* Do these 5 states with a low affordable housing supply also have higher poverty exposure over time?


In [429]:
#Avg poverty share (pct) of ppl experiencing poverty who live in high-poverty neighborhoods
poverty_st_yr = county.groupby(["state_name","year"], as_index=False).agg(avg_poverty= ("share_poverty_exposure","mean")) #adds pop col with mean of pop
poverty_st_yr["avg_poverty"] = poverty_st_yr["avg_poverty"] * 100 #converts these proportions to percentages 0<avg_poverty<100 instead of 0<...<1

#Dropping NA values
poverty_st_yr = poverty_st_yr.dropna()
poverty_st_yr

,state_name,year,avg_poverty
0,Alabama,2014,10.489371
2,Alabama,2016,11.453309
4,Alabama,2018,9.335122
7,Alabama,2021,8.774582
9,Alabama,2023,8.858536
...,...,...,...
501,Wyoming,2014,0.969597
503,Wyoming,2016,0.219548
505,Wyoming,2018,0.200941
508,Wyoming,2021,0.739842


In [430]:
#Using lowest 5 states from prev graph, get subset of poverty exposure d.f.
avg_low2 = poverty_st_yr[poverty_st_yr["state_name"].isin(lowest_5_hous)].sort_values(["state_name", "year"]) #Gets subset of poverty_st_yr d.f. but only selects the states in the top_5

#Plotting Avg poverty exposure for lowest 5 affordability states
fig = px.line(
    avg_low2,
    x="year",
    y="avg_poverty",
    color="state_name",
    markers=True,
    title=("Average Poverty Exposure by State Over Time "
            f"<br><span style='font-size:15px;'>{subtitle}</span>" # subtitle uses HTML
)
)

fig.update_traces(mode="lines+markers", connectgaps=True)  # explicit

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Average Poverty Exposure (%)",
    legend_title_text="State"
)


fig.show()

* This graph uses the 5 states with the lowest average affordable housing units from the previous graph and plots those states by average poverty exposure.
    * The average **poverty exposure** here refers to the percentage of people experiencing poverty that live in neighborhoods of high-poverty.

## Research Question 3

Is there an association between states with more affordable housing shortages and higher poverty exposure?

In [ ]:
#Merging avg affordability share AND avg poverty share
state_combo = pd.merge(avg_low, avg_low2, on=["state_name","year"], how="inner")

state_combo.head()

,state_name,year,avg_afford,avg_poverty
0,California,2014,1.298858,7.495857
1,California,2015,1.338451,NaN
2,California,2016,1.310548,6.912452
3,California,2017,1.292715,NaN
4,California,2018,1.317776,5.624557


In [435]:
#Plotting avg affordable units vs average poverty exposure (across ALL years at once)
fig = px.scatter(
    state_combo,
    x="avg_poverty",
    y="avg_afford",
    hover_name="state_name",
    trendline="ols",
    title="Average Affordable Housing Units vs. Average Poverty Exposure Across All Years"
    f"<br><span style='font-size:15px;'>{subtitle}</span>", # subtitle uses HTML
    trendline_color_override="red"

)


fig.update_layout(
    xaxis_title="Average Poverty Exposure (%)",
    yaxis_title="Average Housing Units per 100 Households",
    legend_title_text="State"
)

fig.show()


* This graph plots the average number of affordable housing units per 100 households by the average percentage of people experiencing poverty who live in high-poverty neighborhoods.

In [ ]:
#Linear Model 1: Does average housing affordability predict average poverty exposure?
aff_pov_subset = state_combo.dropna(subset=["avg_afford", "avg_poverty"]).copy() #Drops NAs in the copy

X = sm.add_constant(aff_pov_subset["avg_afford"])
y = aff_pov_subset["avg_poverty"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            avg_poverty   R-squared:                       0.167
Model:                            OLS   Adj. R-squared:                  0.131
Method:                 Least Squares   F-statistic:                     4.622
Date:                Thu, 18 Dec 2025   Prob (F-statistic):             0.0423
Time:                        02:38:42   Log-Likelihood:                -73.621
No. Observations:                  25   AIC:                             151.2
Df Residuals:                      23   BIC:                             153.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         52.8758     21.371      2.474      0.0

* The above model uses average housing affordability as the predictor variable and average poverty exposure as the response variable.
    * Does average housing affordability predict average poverty exposure?
* The summary results show that the p value `P >|t| = 0.042 < 0.05` for the average affordability variable `avg_afford` which lets us know that there is a statistically significant linear association between average affordability and average poverty exposure in this sample of state-year averages.
* The `R-squared=0.167` value means that affordability alone explains 16.7% of the variation in poverty exposure across these observations.
    * This suggests the relationship exists but not very strongly and there could be other county/state conditions that influence poverty exposure.

## Research Question 4

Is lower affordable housing availability associated with higher rates of student homelessness across counties?


In [ ]:
#Avg share (pct) of public-school children who are ever homeless during the school year
homeless_st_yr = county.groupby(["state_name","year"], as_index=False).agg(avg_homeless= ("share_homeless","mean")) #adds pop col with mean of pop
homeless_st_yr["avg_homeless"] = homeless_st_yr["avg_homeless"] * 100 #converts these proportions to percentages 0<avg_poverty<100 instead of 0<...<1
homeless_st_yr

,state_name,year,avg_homeless
0,Alabama,2014,2.748834
1,Alabama,2015,2.565661
2,Alabama,2016,2.574455
3,Alabama,2017,2.066709
4,Alabama,2018,2.190347
...,...,...,...
506,Wyoming,2019,2.087379
507,Wyoming,2020,2.217973
508,Wyoming,2021,2.218168
509,Wyoming,2022,2.485686


In [ ]:
#Gets those lowest avg affordability states from earlier and gets subset of this dataset with the avg_homeless variable
avg_low3 = homeless_st_yr[homeless_st_yr["state_name"].isin(lowest_5_hous)].sort_values(["state_name", "year"]) #Gets subset of poverty_st_yr d.f. but only selects the states in the top_5

In [ ]:
#Putting avg_homeless into the dataset with avg_poverty and avg_afford
state_combo2 = pd.merge(state_combo, avg_low3, on=["state_name","year"], how="inner")

state_combo2.head()

,state_name,year,avg_afford,avg_poverty,avg_homeless
0,California,2014,1.298858,7.495857,3.877171
1,California,2015,1.338451,NaN,4.569518
2,California,2016,1.310548,6.912452,4.349077
3,California,2017,1.292715,NaN,4.874668
4,California,2018,1.317776,5.624557,5.207350


In [439]:
#Plotting Average poverty exposure by Average share of homeless public-school children
fig = px.scatter(
    state_combo2,
    x="avg_afford",
    y="avg_homeless",
    hover_name="state_name",
    trendline="ols",
    title="Average Share of Homeless Students vs. Average Affordable Housing Units"
    f"<br><span style='font-size:15px;'>{subtitle}</span>",
    trendline_color_override="purple"
)

fig.update_layout(
    xaxis_title="Average Affordable Housing Units per 100 Households",
    yaxis_title="Average Share of Homeless Students (%)",
    legend_title_text="State"
)

fig.show()


* This graph plots the average percentage of public-school children who are ever homeless during the school year against the average number of affordable housing units per 100 households.

In [ ]:
#Linear Model 2: How does average affordability predict average
df = state_combo2.dropna(subset=["avg_afford", "avg_homeless"]).copy()

# Predictors: affordability + homelessness
X = df["avg_afford"]
X = sm.add_constant(X)

# Response: poverty exposure
y = df["avg_homeless"]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           avg_homeless   R-squared:                       0.333
Model:                            OLS   Adj. R-squared:                  0.315
Method:                 Least Squares   F-statistic:                     18.95
Date:                Thu, 18 Dec 2025   Prob (F-statistic):           9.76e-05
Time:                        02:31:26   Log-Likelihood:                -82.524
No. Observations:                  40   AIC:                             169.0
Df Residuals:                      38   BIC:                             172.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         30.8191      6.284      4.904      0.0

* This linear regression model uses average housing affordability as the predictor variable and average share of homeless students as the response variable.
    * How does average housing affordability predict the average share of homeless public-school students?
*The summary results from the model show that the p-value `P>|t|<0.05` for the average affordability variable `avg_afford` which lets us know that there is a statistically significant linear association between average affordability and the average share of homeless students.
* The `R-squared=0.33` value means that affordability explains 33% of the variation in student homelessness across these observations.
    * This suggests a moderate correlation between the two variables.

## Conclusions

This longitudinal analysis demonstrates:
* Data wrangling with real longitudinal indicators


* State-by-year aggregation

* Clear trend visualization using Plotly line charts

* Exploratory association testing using scatterplots with fitted regression lines and simple linear regression (OLS).
